# 02 — Table d'analyse et construction de la cible

Objectif : une ligne par participant, des variables lisibles, et la cible
« syndrome métabolique ». Le détail est dans `src/preprocessing.py` et `src/target.py` ;
ici je montre les décisions et je vérifie les résultats.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60)
from src import data_loader, preprocessing, target
from src.config import COMPONENTS, CYCLES
tables = data_loader.load_all(list(COMPONENTS), list(CYCLES))

## Recodage des non-réponses

NHANES code « Refus » et « Ne sait pas » par des nombres (7, 9, 77, 99, 7777…) qui
dépendent de la variable. Laissés tels quels ils polluent tout. La correspondance est
dans `config.SPECIAL_MISSING` et `_blank_special()` les passe en `NaN`.

Un piège plus sournois : après lecture du `.XPT`, certaines valeurs 0 ressortent en
`5.4e-79` (artefact d'encodage). Ça concerne surtout la fréquence de consommation
d'alcool. `_fix_xpt_zeros()` les recale à 0.

In [2]:
alq = tables["ALQ"]
alq[alq["cycle"] == "2017-2018"]["ALQ121"].value_counts().head(3)

ALQ121
5.397605e-79    1049
6.000000e+00     606
1.000000e+01     527
Name: count, dtype: int64

## Harmonisation alcool et sommeil

Ce sont les deux endroits où les cycles ne sont pas alignés.

**Alcool.** En 2013-2016 : une question de fréquence (`ALQ120Q` + unité `ALQ120U`) et une
de quantité (`ALQ130`). En 2017-2018 : une fréquence déjà codée en classes (`ALQ121`) et
`ALQ130`. Je ramène tout à une même grandeur, **verres par semaine estimés**, en
convertissant les classes 2017-2018 en occasions par an (0 = jamais, 1 = tous les jours,
… 10 = 1-2 fois/an) puis en multipliant par la quantité par occasion. C'est une
estimation, pas une mesure — mais elle est cohérente entre cycles, ce qui est le but.

**Sommeil.** `SLD010H` (heures entières, 2013-2014) devient `SLD012` (demi-heures
possibles) ensuite. Je prends l'un ou l'autre selon ce qui existe.

In [3]:
df = preprocessing.assemble(tables)
print("table assemblée :", df.shape)
df[["age", "sexe", "alcool_verres_sem", "sommeil_h", "activite_met_min_sem",
    "fibres_g", "sucres_g", "sodium_mg"]].describe().round(1)

table assemblée : (29400, 54)


,age,alcool_verres_sem,sommeil_h,activite_met_min_sem,fibres_g,sucres_g,sodium_mg
count,29400.0,13648.0,18861.0,24595.0,24342.0,24342.0,24342.0
mean,32.5,3.1,7.4,808.1,15.3,102.3,3100.8
std,24.9,7.8,1.6,1727.6,9.0,59.6,1545.6
min,0.0,0.0,2.0,0.0,0.0,0.0,0.0
25%,10.0,0.0,6.5,0.0,9.3,62.5,2080.1
50%,28.0,0.4,7.5,0.0,13.6,91.4,2882.8
75%,54.0,2.9,8.0,960.0,19.4,128.0,3860.0
max,80.0,126.0,14.5,27440.0,105.8,1115.5,25949.0


## Population d'analyse

Quatre filtres, dans cet ordre :

1. adultes (≥ 20 ans) — les seuils du syndrome métabolique sont ceux de l'adulte ;
2. hors grossesse ;
3. sous-échantillon à jeun — sinon pas de glycémie ni de triglycérides ;
4. **hors diabète et hors maladie cardiovasculaire déjà diagnostiqués**. C'est un choix :
   je veux prédire un *risque* chez des gens a priori sains, pas retrouver une maladie
   connue. Garder les diabétiques gonflerait artificiellement le lien
   glycémie ↔ syndrome et rendrait le modèle moins utile.

In [4]:
filt, journal = preprocessing.filtre_population(df)
journal

,étape,exclus,restants
0,départ,29400,29400
1,âge ≥ 20 ans,12343,17057
2,hors grossesse,190,16867
3,sous-échantillon à jeun (glycémie + TG),9737,7130
4,hors diabète / MCV déjà diagnostiqués,1609,5521


In [5]:
y = target.syndrome_metabolique(filt)
crit = target.criteres(filt)

print("cible :", y.value_counts(dropna=False).to_dict())
print(f"prévalence : {y.mean():.1%}   (NaN = trop de critères manquants pour trancher)")
print()
print("part de participants remplissant chaque critère :")
for nom, part in crit.mean().sort_values(ascending=False).items():
    print(f"  {nom:10s} {part:5.1%}")

cible : {0.0: 3647, 1.0: 1758, nan: 116}
prévalence : 32.5%   (NaN = trop de critères manquants pour trancher)

part de participants remplissant chaque critère :
  taille     53.4%
  glycemie   50.9%
  tension    40.6%
  hdl        25.9%
  trigly     19.2%


Une prévalence autour de **32 %** est cohérente avec ce que la littérature rapporte pour
les adultes américains sur cette période. Les critères les plus fréquents sont le tour de
taille et la glycémie ; les triglycérides élevés sont les plus rares.

Note sur un choix de définition : le critère « tension » et le critère « glycémie »
comptent aussi comme remplis si la personne est **traitée** pour ça (antihypertenseur,
antidiabétique), même si la valeur mesurée est normale — c'est la définition harmonisée
de 2009. Ça évite de classer « sain » quelqu'un dont la tension n'est normale que grâce
au traitement.

In [6]:
out = filt.assign(cible=y)
out.to_parquet(ROOT / "data" / "processed" / "analytique.parquet")
print("écrit :", out.shape, "->", "data/processed/analytique.parquet")
print("lignes exploitables (cible connue) :", int(out["cible"].notna().sum()))

écrit : (5521, 55) -> data/processed/analytique.parquet
lignes exploitables (cible connue) : 5405
